# Обзор девяти задач регрессии

В каждой волне четыре семейства обучаются на полном наборе входных признаков каждой задачи. Для Gas Turbine отдельно оцениваются CO и NOx. Используется нулевое разбиение каждого датасета и пара seed (11, 101). Для SARCOS сохраняется официальный тест, для временных рядов — временное разбиение.

Отбор лучшей эпохи идёт по валидационному RMSE за 100 эпох Adam. Тестовая выборка остаётся отложенной. Сравнивай ошибки внутри задачи: единицы измерения между датасетами различаются. Для запуска укажи GPU и выполни **Run All**, либо остановись после таблицы первой волны.

In [ ]:
PROJECT_DIR = None
DEVICE = "cuda:1"  # Выбери свободную GPU в окружении ядра.
VERSION = "survey-v1"
DOWNLOAD = True  # Скачать только отсутствующие исходные файлы.
PARALLEL_TRIALS = 4
GPU_BUDGET_MIB = 4096

In [ ]:
import sys
from pathlib import Path

if PROJECT_DIR is None:
    cwd = Path.cwd().resolve()
    checkout = next(
        (folder for folder in (cwd, *cwd.parents) if (folder / "src/experiments/runner.py").is_file()),
        None,
    )
else:
    checkout = Path(PROJECT_DIR).expanduser()
    if not checkout.is_absolute() or not (checkout / "src/experiments/runner.py").is_file():
        raise ValueError("Укажи в PROJECT_DIR абсолютный путь к корню mgs-kan на сервере.")
    checkout = checkout.resolve()

if checkout is not None and sys.path[0:1] != [str(checkout)]:
    sys.path.insert(0, str(checkout))

try:
    from src.paths import DATA_DIR, ENV_FILE, PROJECT_ROOT
except ModuleNotFoundError as error:
    raise ImportError("Укажи PROJECT_DIR и выполни ячейки заново.") from error

if checkout is not None and PROJECT_ROOT != checkout:
    raise RuntimeError("Ядро уже загрузило src из другой папки. Перезапусти ядро.")
if not (PROJECT_ROOT / "notebooks/baselines/run_series.ipynb").is_file():
    raise RuntimeError("Укажи PROJECT_DIR с рабочей копией mgs-kan и перезапусти ядро.")
print(f"Проект: {PROJECT_ROOT}\nНастройки: {ENV_FILE}\nДанные: {DATA_DIR}")

In [ ]:
import pandas as pd
from IPython.display import display

from src.config import MonitorSettings, ResourceSettings, TrackingSettings
from src.experiments.search import ValidationGrid

survey = ValidationGrid.preset("benchmark_survey")
resources = ResourceSettings(_env_file=ENV_FILE, max_parallel_trials=PARALLEL_TRIALS,
                             cpu_threads=1, gpu_budget_mib=GPU_BUDGET_MIB)
tracking = TrackingSettings(_env_file=ENV_FILE)
monitoring = MonitorSettings(_env_file=ENV_FILE)
output = PROJECT_ROOT / "results" / "benchmark-survey" / VERSION
print(f"{len(survey.datasets)} задач · {survey.wave(1).size} + {survey.wave(2).size} комбинаций")
print(f"Устройство: {DEVICE}. Каталог результатов: {output}")

| Семейство | Волна 1 | Волна 2 |
|---|---|---|
| MGS-KAN | 7 одиночных, 4 парных ядра | 16 одиночных, 8 парных ядер |
| MLP | 2 слоя с бюджетом параметров MGS-KAN первой волны | 2 слоя с бюджетом параметров MGS-KAN второй волны |
| Deep MLP | 4 слоя × 128 | 6 слоёв × 256 |
| ResNet | 4 блока × 128 | 6 блоков × 256 |

Для Superconductivity профиль запускает по одной модели за раз при том же бюджете VRAM. В текущем эксперименте MLflow проверяются завершённые валидационные запуски, включая результаты из других серий. При нескольких совпадениях берётся самый свежий подходящий запуск с чекпоинтом. Повторный запуск ячеек заново проверяет историю.

In [ ]:
def run_wave(number):
    wave = survey.wave(number)
    runs = wave.run(version=f"{VERSION}-w{number}", root=DATA_DIR, download=DOWNLOAD,
                    device=DEVICE, resources=resources, tracking=tracking, monitoring=monitoring)
    summary = runs[["dataset", "model", "val_rmse", "val_r2", "parameters", "seconds",
                    "best_epoch", "reused", "run_id"]].sort_values(["dataset", "val_rmse"])
    output.mkdir(parents=True, exist_ok=True)
    runs.to_csv(output / f"wave-{number}-runs.csv", index=False)
    summary.to_csv(output / f"wave-{number}-summary.csv", index=False)
    reused = int(runs.reused.sum())
    print(f"Волна {number}: {len(runs)}/{wave.size}. Из MLflow: {reused}. Новых: {len(runs) - reused}.")
    print("Валидационный RMSE по задачам и моделям")
    display(summary.pivot(index="dataset", columns="model", values="val_rmse").round(4))
    with pd.option_context("display.max_rows", len(summary)):
        display(summary.round({"val_rmse": 4, "val_r2": 4, "seconds": 1}))
    return runs

## Волна 1 · базовые конфигурации

In [ ]:
first = run_wave(1)

## Волна 2 · увеличенные конфигурации

In [ ]:
second = run_wave(2)

## Изменение ошибки при увеличении модели

Отрицательная разница RMSE означает улучшение во второй волне. На задачу приходится одна пара seed, поэтому порядок моделей предварительный.

In [ ]:
keys = ["dataset", "model"]
comparison = first[keys + ["val_rmse"]].merge(
    second[keys + ["val_rmse"]], on=keys, suffixes=("_wave1", "_wave2"), validate="one_to_one",
)
comparison["delta_rmse"] = comparison.val_rmse_wave2 - comparison.val_rmse_wave1
comparison.to_csv(output / "wave-comparison.csv", index=False)
display(comparison.pivot(index="dataset", columns="model", values="delta_rmse").round(4))